In [2]:
import os

def load_emails(directory, files):
    emails = []
    for filename in files:
        filepath = os.path.join(directory, filename)
        with open(filepath, "rb") as f:
            emails.append(f.read())
    return emails

ham_emails = load_emails("dataset/20021010_easy_ham/easy_ham/", easyham_files)
spam_emails = load_emails("dataset/20021010_spam/spam/", spam_files)

NameError: name 'easyham_files' is not defined

In [ ]:
import email

def parse_email(raw_email):
    return email.message_from_bytes(raw_email)

ham_parsed = [parse_email(e) for e in ham_emails]
spam_parsed = [parse_email(e) for e in spam_emails]

msg = ham_parsed[0]
print(msg["subject"])
print(msg.get_payload())

In [ ]:
def get_email_body(msg):
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                return part.get_payload(decode=True).decode("utf-8", errors="ignore")
    else:
        return msg.get_payload(decode=True).decode("utf-8", errors="ignore")

In [ ]:
ham_bodies = [get_email_body(e) for e in ham_parsed]
spam_bodies = [get_email_body(e) for e in spam_parsed]

import numpy as np
X = ham_bodies + spam_bodies
y = np.array([0] * len(ham_bodies) + [1] * len(spam_bodies))

In [ ]:
X[0]

In [ ]:
y

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")

In [ ]:
valid = [(body, label) for body, label in zip(X, y) if body is not None]
X = [v[0] for v in valid]
y = np.array([v[1] for v in valid])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LogisticRegression())
])

pipeline.fit(X_train, y_train)

In [ ]:
y_hat = pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test,y_hat)
cm